# Prototipo: clustering de telemetria GPS (Artisa)

Este notebook reproduce, paso a paso y con la MISMA logica, lo que hacen `modelo_machine/features.py` y `modelo_machine/telemetry.py` en el proyecto. La idea: experimentar aqui con los datos reales, y cuando quede bien, copiar las funciones tal cual de vuelta a esos dos archivos (los nombres de funcion son identicos a proposito).

**Pipeline real del proyecto:** MongoDB (`gps_data.raw_gps`) -> ETL -> MariaDB (`raw_gps`) -> este modelo lee de MariaDB.

**Aqui:** corre local en VS Code (kernel Jupyter/ipykernel), leyendo una muestra exportada a CSV en vez de conectarse directo a Mongo/MariaDB. Tambien puedes subirlo a Google Colab sin cambios (usa `files.upload()` ahi en vez de leer el CSV local).

## Hoja de ruta ML — estado actual

Leyenda: ✅ hecho · 🟡 parcial (falta robustecer) · ⬜ pendiente

Los titulos "Paso N" de este notebook coinciden con el numero de esta tabla.

| # | Paso | Estado | Donde esta / que falta |
|---|------|--------|------------------------|
| 1 | Definir problema y objetivo | 🟡 | El Paso 7 dice "zonas frecuentes", pero falta definir el objetivo de negocio y como medir el exito |
| 2 | Recoleccion de datos | ✅ | Seccion Paso 2 (MariaDB). Ojo: solo toma las ultimas `MAX_ROWS` filas, revisar representatividad |
| 3 | Analisis exploratorio (EDA) | ⬜ | Solo hay un scatter. Faltan distribuciones, nulos, outliers, correlaciones, reportes por vehiculo/hora |
| 4 | Limpieza y preprocesamiento | 🟡 | Seccion Paso 4 y 5: `clean_rows`, `impute_missing`, `scale_numeric`. Falta filtrar coordenadas invalidas/duplicadas y outliers de lat/lon |
| 5 | Ingenieria de caracteristicas | 🟡 | Seccion Paso 4 y 5: `hora_dia`, `exceso_velocidad`, one-hot de `event_name`. Faltan ventanas de tiempo, dia de la semana, agregaciones por vehiculo |
| 6 | Division train / validation / test | ⬜ | Se entrena con todos los datos. Dividir cronologicamente para evitar fuga de datos |
| 7 | Seleccion y entrenamiento de modelos | 🟡 | Seccion Paso 7: solo DBSCAN. Falta un modelo base y comparar KMeans, HDBSCAN, GMM |
| 8 | Evaluacion con metricas | 🟡 | Solo `cobertura_pct` y `puntos_ruido`. Faltan silhouette, Davies-Bouldin, Calinski-Harabasz |
| 9 | Validacion y ajuste de hiperparametros | ⬜ | `radius_km` / `min_samples` se ajustan a mano. Falta busqueda sistematica (grid / Optuna) y prueba de estabilidad |
| 10 | Interpretacion de resultados | 🟡 | Seccion Paso 10: tabla de zonas, scatter, mapa folium y geocercas de faenas (poligonos reales). Falta perfilar cada zona (horarios, exceso de velocidad, tipo de evento) |
| 11 | Despliegue | 🟡 | Seccion Paso 11: ya existe el servicio en `modelo_machine`. Falta guardar el pipeline/modelo serializado |
| 12 | Monitoreo y mantenimiento | ⬜ | Sin deteccion de data drift, reentrenamiento programado ni versionado de modelos |

Ir actualizando esta tabla a medida que se completen los pasos.

## Paso 2 — Recolección de datos (conexión a MariaDB)

Sin CSV ni Mongo: la celda de abajo lee la tabla `raw_gps` de MariaDB, que es donde el ETL ya dejo los datos (igual que hace `modelo_machine`).

Requisitos:

1. `artisa-mariadb` corriendo y `healthy` (`docker compose ps`). El puerto 3306 ya esta publicado en `127.0.0.1` en `compose.yaml`.
2. Las credenciales salen del `.env` de la raiz (`MARIADB_USER`, `MARIADB_PASSWORD`, `MARIADB_DATABASE`).
3. `pymysql` instalado en el `.venv`.
4. Si sale `Access denied`, la contraseña del `.env` no coincide con la que tiene el volumen de MariaDB; hay que corregirla antes de seguir.

In [ ]:
# Si falta algun paquete en tu entorno local: pip install pymysql pandas numpy scikit-learn matplotlib

import json
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymysql
from sklearn.cluster import DBSCAN

In [ ]:
def load_env(path):
    """Parser minimo de .env (KEY=VALUE), sin dependencias extra."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env = load_env(Path("..") / ".env")

connection = pymysql.connect(
    host="127.0.0.1",  # publicado por compose.yaml; DB_HOST=mariadb solo aplica dentro de la red de Docker
    port=3306,
    user=env["MARIADB_USER"],
    password=env["MARIADB_PASSWORD"],
    database=env["MARIADB_DATABASE"],
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor,
)

MAX_ROWS = 20000  # mismo default que TELEMETRY_MODEL_MAX_ROWS en compose.yaml

with connection, connection.cursor() as cursor:
    cursor.execute(
        """SELECT report_id, report_date, asset_id, asset_name, event_name,
           latitude, longitude, speed, cartography_limit_speed
           FROM raw_gps ORDER BY report_date DESC LIMIT %s""",
        (MAX_ROWS,),
    )
    df = pd.DataFrame(cursor.fetchall())

print(df.shape)
df.head()

## Paso 4 y 5 — Limpieza / preprocesamiento e ingeniería de características (identico a `modelo_machine/features.py`)

Las mismas 5 funciones, mismos nombres, misma firma (reciben/devuelven listas de dicts, no DataFrame, para que el copy-paste al proyecto sea literal).

- **Paso 4 (limpieza y preprocesamiento):** `clean_rows`, `impute_missing`, `scale_numeric`.
- **Paso 5 (ingeniería de características):** `derive_features`, `encode_categorical`.

In [ ]:
NUMERIC_FEATURES = ("latitude", "longitude", "speed", "cartography_limit_speed",
                     "exceso_velocidad", "hora_dia")
SPEED_MIN_KMH = 0.0
SPEED_MAX_KMH = 200.0


def clean_rows(rows, speed_min=SPEED_MIN_KMH, speed_max=SPEED_MAX_KMH):
    """Descarta reportes con velocidad fuera de un rango fisico razonable."""
    cleaned = []
    for row in rows:
        speed = row.get("speed")
        if speed is not None and not (speed_min <= float(speed) <= speed_max):
            continue
        cleaned.append(row)
    return cleaned


def impute_missing(rows):
    """Rellena speed y cartography_limit_speed nulos con la mediana del lote."""
    speeds = [float(row["speed"]) for row in rows if row.get("speed") is not None]
    limits = [float(row["cartography_limit_speed"]) for row in rows
              if row.get("cartography_limit_speed") is not None]
    speed_median = statistics.median(speeds) if speeds else 0.0
    limit_median = statistics.median(limits) if limits else 0.0

    imputed = []
    for row in rows:
        row = dict(row)
        if row.get("speed") is None:
            row["speed"] = speed_median
        if row.get("cartography_limit_speed") is None:
            row["cartography_limit_speed"] = limit_median
        imputed.append(row)
    return imputed


def derive_features(rows):
    """Agrega hora_dia (0-23) y exceso_velocidad a cada fila."""
    derived = []
    for row in rows:
        row = dict(row)
        report_date = row.get("report_date")
        row["hora_dia"] = report_date.hour if isinstance(report_date, datetime) else 0
        row["exceso_velocidad"] = float(row["speed"]) - float(row["cartography_limit_speed"])
        derived.append(row)
    return derived


def encode_categorical(rows, column="event_name"):
    """One-hot encoding dinamico: una columna nueva por cada categoria vista en el lote."""
    categories = sorted({row.get(column) or "desconocido" for row in rows})
    category_columns = [f"{column}_{category}" for category in categories]

    encoded = []
    for row in rows:
        row = dict(row)
        value = row.get(column) or "desconocido"
        for category, category_column in zip(categories, category_columns):
            row[category_column] = 1.0 if value == category else 0.0
        encoded.append(row)
    return encoded, category_columns


def scale_numeric(rows, columns):
    """Estandarizacion z-score: (valor - media) / desviacion estandar."""
    scaled = [dict(row) for row in rows]
    for column in columns:
        values = [float(row[column]) for row in scaled]
        mean = statistics.fmean(values)
        stdev = statistics.pstdev(values) or 1.0
        for row in scaled:
            row[column] = (float(row[column]) - mean) / stdev
    return scaled


def build_dataset(rows, categorical_column="event_name"):
    """Orquesta limpieza -> imputacion -> features derivadas -> encoding -> escalado.

    Devuelve (filas_transformadas, columnas_numericas_finales).
    """
    rows = clean_rows(rows)
    rows = impute_missing(rows)
    rows = derive_features(rows)
    rows, category_columns = encode_categorical(rows, categorical_column)
    rows = scale_numeric(rows, list(NUMERIC_FEATURES))
    return rows, list(NUMERIC_FEATURES) + category_columns

## Paso 4 y 5 (cont.) — Aplicar el pipeline y construir la matriz X

Convertimos el DataFrame a lista de dicts (lo que espera `build_dataset`), corremos el pipeline, y armamos la matriz `X` (numpy) lista para cualquier modelo.

In [ ]:
rows_in = df.to_dict(orient="records")
rows_out, columnas = build_dataset(rows_in)

X = np.array([[row[col] for col in columnas] for row in rows_out], dtype=float)
print("Matriz X:", X.shape, "columnas:", columnas)
X[:5]

## Paso 7 — Selección y entrenamiento del modelo: DBSCAN (identico a `modelo_machine/telemetry.py::train_hotspots`)

Ojo: el clustering de zonas usa **lat/lon SIN escalar** (necesita coordenadas reales en radianes para la metrica haversine), no la matriz `X` del paso anterior. `X` queda lista para otros modelos futuros (deteccion de anomalias de velocidad, prediccion de ETA) que si se benefician del escalado.

In [ ]:
EARTH_RADIUS_KM = 6371.0088


def train_hotspots(rows, radius_km=0.8, min_samples=5):
    """Agrupa posiciones cercanas sin requerir rutas etiquetadas."""
    if len(rows) < max(10, min_samples * 2):
        raise ValueError("Se necesitan al menos 10 posiciones para entrenar telemetria")
    coordinates = np.radians([[float(row["latitude"]), float(row["longitude"])] for row in rows])
    labels = DBSCAN(
        eps=float(radius_km) / EARTH_RADIUS_KM,
        min_samples=int(min_samples),
        metric="haversine",
        algorithm="ball_tree",
    ).fit_predict(coordinates)

    grouped = defaultdict(list)
    noise = 0
    for row, label in zip(rows, labels):
        if int(label) == -1:
            noise += 1
        else:
            grouped[int(label)].append(row)

    zones = []
    for label, members in grouped.items():
        speeds = [float(row["speed"]) for row in members if row.get("speed") is not None]
        zones.append({
            "zona_id": label,
            "latitud": round(sum(float(row["latitude"]) for row in members) / len(members), 6),
            "longitud": round(sum(float(row["longitude"]) for row in members) / len(members), 6),
            "reportes": len(members),
            "vehiculos": len({str(row["asset_id"]) for row in members}),
            "velocidad_promedio": round(sum(speeds) / len(speeds), 2) if speeds else None,
        })
    zones.sort(key=lambda zone: (-zone["reportes"], zone["zona_id"]))
    metrics = {
        "reportes_entrenamiento": len(rows),
        "vehiculos": len({str(row["asset_id"]) for row in rows}),
        "zonas_detectadas": len(zones),
        "puntos_ruido": noise,
        "cobertura_pct": round((len(rows) - noise) * 100 / len(rows), 2),
        "radio_km": float(radius_km),
        "min_muestras": int(min_samples),
        "alcance": "Clustering geoespacial no supervisado; identifica zonas frecuentes, no rutas etiquetadas.",
    }
    return metrics, zones[:250]


# Mismos defaults que TELEMETRY_MODEL_RADIUS_KM / TELEMETRY_MODEL_MIN_SAMPLES en compose.yaml
metrics, zones = train_hotspots(rows_in, radius_km=0.8, min_samples=5)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## Paso 10 — Interpretación de resultados: zonas detectadas

In [ ]:
zonas_df = pd.DataFrame(zones)
zonas_df.head(20)

## Paso 10 (cont.) — Interpretación de resultados: visualización (scatter simple; cambia radius_km / min_samples arriba y vuelve a correr para comparar)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["longitude"], df["latitude"], s=2, alpha=0.2, label="reportes")
if len(zonas_df):
    plt.scatter(zonas_df["longitud"], zonas_df["latitud"], s=zonas_df["reportes"] / 5,
                c="red", alpha=0.6, label="zonas detectadas")
plt.xlabel("longitud")
plt.ylabel("latitud")
plt.legend()
plt.title("Zonas frecuentes detectadas por DBSCAN")
plt.show()

In [ ]:
# Mapa interactivo. Requiere: pip install folium  (y internet para los tiles de OpenStreetMap)
import folium

centro = [df["latitude"].mean(), df["longitude"].mean()]
mapa = folium.Map(location=centro, zoom_start=6, tiles="OpenStreetMap")

# Muestra de reportes (2.000 max para que el mapa no se ponga pesado)
for _, fila in df.sample(min(2000, len(df)), random_state=0).iterrows():
    folium.CircleMarker(
        [fila["latitude"], fila["longitude"]],
        radius=1, color="#1f77b4", opacity=0.4, fill=True,
    ).add_to(mapa)

# Zonas detectadas: circulo del radio del modelo (radius_km), con datos al hacer clic
for zona in zones:
    folium.Circle(
        [zona["latitud"], zona["longitud"]],
        radius=metrics["radio_km"] * 1000,
        color="red", fill=True, fill_opacity=0.25,
        popup=(f"Zona {zona['zona_id']}<br>reportes: {zona['reportes']}<br>"
               f"vehiculos: {zona['vehiculos']}<br>vel. promedio: {zona['velocidad_promedio']}"),
    ).add_to(mapa)

mapa

## Paso 10 (cont.) — Interpretación de resultados: geocercas de las faenas sobre el mapa

Lee la tabla `faenas` directo de MariaDB (misma conexión del Paso 2, sin CSV intermedio) y dibuja cada faena como un polígono real, no un círculo aproximado.

In [ ]:
with pymysql.connect(
    host="127.0.0.1", port=3306,
    user=env["MARIADB_USER"], password=env["MARIADB_PASSWORD"], database=env["MARIADB_DATABASE"],
    charset="utf8mb4", cursorclass=pymysql.cursors.DictCursor,
) as faenas_connection, faenas_connection.cursor() as faenas_cursor:
    faenas_cursor.execute(
        "SELECT id, nombre, area_m2, ST_AsText(geocerca) AS poligono_wkt FROM faenas ORDER BY id"
    )
    faenas = faenas_cursor.fetchall()

for faena in faenas:
    # 'POLYGON((lon lat,lon lat,...))' -> lista de puntos (lat, lon) en orden
    puntos_lonlat = faena["poligono_wkt"][len("POLYGON(("):-2].split(",")
    puntos = [(float(lat), float(lon)) for lon, lat in
              (par.strip().split(" ") for par in puntos_lonlat)]
    folium.Polygon(
        locations=puntos,
        color="green",
        weight=2,
        fill=True,
        fill_opacity=0.2,
        popup=f"{faena['nombre']}<br>area: {faena['area_m2']:.0f} m2",
    ).add_to(mapa)

mapa

## Paso 11 — Despliegue: como portar esto al proyecto

1. Ajusta `radius_km` / `min_samples` en el Paso 7 hasta que las zonas tengan sentido para tu operacion (mas reportes = zonas mas finas o mas gruesas).
2. Las funciones `clean_rows`, `impute_missing`, `derive_features`, `encode_categorical`, `scale_numeric`, `build_dataset` son copy-paste directo a `modelo_machine/features.py` (son literalmente el mismo codigo).
3. `train_hotspots` es copy-paste directo a `modelo_machine/telemetry.py` (mismo codigo, ya esta ahi).
4. Si cambias algo aqui (ej. agregar una feature nueva, otro algoritmo), replica el cambio en esos dos archivos del proyecto y ajusta las variables de entorno relevantes en `compose.yaml` si aplica (`TELEMETRY_MODEL_RADIUS_KM`, `TELEMETRY_MODEL_MIN_SAMPLES`, `TELEMETRY_MODEL_MAX_ROWS`).